# Import Libraries

In [18]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm


# Device Load

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


# Normalization

In [20]:
def normalize_img(img):
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img


# Final ISLES Dataset Indexing

In [21]:
class ISLESDataset2D(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.index = []

        for case in sorted(os.listdir(root_dir)):
            if not case.startswith("sub-strokecase"):
                continue

            mask_path = os.path.join(
                root_dir, "derivatives", case, "ses-0001",
                f"{case}_ses-0001_msk.nii.gz"
            )

            if not os.path.exists(mask_path):
                continue

            mask_vol = nib.load(mask_path).get_fdata()

            for z in range(mask_vol.shape[2]):
                if np.sum(mask_vol[:, :, z]) > 0:
                    self.index.append((case, z))

        print("Indexed slices:", len(self.index))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        case, z = self.index[idx]

        dwi = nib.load(
            os.path.join(
                self.root_dir, case, "ses-0001", "dwi",
                f"{case}_ses-0001_dwi.nii.gz"
            )
        ).get_fdata()

        adc = nib.load(
            os.path.join(
                self.root_dir, case, "ses-0001", "dwi",
                f"{case}_ses-0001_adc.nii.gz"
            )
        ).get_fdata()

        mask = nib.load(
            os.path.join(
                self.root_dir, "derivatives", case, "ses-0001",
                f"{case}_ses-0001_msk.nii.gz"
            )
        ).get_fdata()

        img = np.stack(
            [normalize_img(dwi[:, :, z]), normalize_img(adc[:, :, z])],
            axis=0
        )

        mask = mask[:, :, z]

        img = torch.from_numpy(img).float().unsqueeze(0)      # (1,2,H,W)
        mask = torch.from_numpy(mask).long().unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

        img = F.interpolate(img, size=(256, 256), mode="bilinear", align_corners=False)
        mask = F.interpolate(mask.float(), size=(256, 256), mode="nearest").long()

        return img.squeeze(0), mask.squeeze(0).squeeze(0)


# Data Load

In [22]:
root_dir = "D:/Capstone/Experiment 4/Datasets/ISLES-2022"

train_dataset = ISLESDataset2D(root_dir)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,      # 🔥 CRITICAL FIX FOR WINDOWS
    pin_memory=True
)


Indexed slices: 4827


In [23]:
print("Dataset length:", len(train_dataset))
print("First index sample:", train_dataset.index[0])


Dataset length: 4827
First index sample: ('sub-strokecase0001', 4)


In [24]:
img, mask = train_dataset[0]
print(img.shape)
print(mask.shape)


torch.Size([2, 256, 256])
torch.Size([256, 256])


# Attention U-Net (Correct Architecture)

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Conv2d(F_g, F_int, 1)
        self.W_x = nn.Conv2d(F_l, F_int, 1)
        self.psi = nn.Conv2d(F_int, 1, 1)

    def forward(self, g, x):
        g = F.interpolate(g, size=x.shape[2:], mode="bilinear", align_corners=False)
        psi = torch.sigmoid(self.psi(F.relu(self.W_g(g) + self.W_x(x))))
        return x * psi


class AttentionUNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=2, base=64):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)

        self.att2 = AttentionGate(base * 4, base * 2, base)
        self.att1 = AttentionGate(base * 2, base, base // 2)

        self.dec2 = DoubleConv(base * 6, base * 2)
        self.dec1 = DoubleConv(base * 3, base)

        self.outc = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(F.max_pool2d(x1, 2))
        x3 = self.enc3(F.max_pool2d(x2, 2))

        x2_att = self.att2(x3, x2)
        x3_up = F.interpolate(x3, size=x2.shape[2:], mode="bilinear", align_corners=False)
        x = self.dec2(torch.cat([x3_up, x2_att], dim=1))

        x1_att = self.att1(x, x1)
        x_up = F.interpolate(x, size=x1.shape[2:], mode="bilinear", align_corners=False)
        x = self.dec1(torch.cat([x_up, x1_att], dim=1))

        return self.outc(x)


In [11]:
model = AttentionUNet()
print(model)


AttentionUNet(
  (enc1): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(128, 256, ker

In [12]:
img, mask = train_dataset[0]
print(img.shape)
print(mask.shape)


torch.Size([2, 256, 256])
torch.Size([256, 256])


# Loss + Optimizer

In [26]:
ce_loss = nn.CrossEntropyLoss()

def dice_loss(preds, targets, smooth=1e-5):
    preds = torch.softmax(preds, dim=1)
    targets_onehot = torch.zeros_like(preds)
    targets_onehot.scatter_(1, targets.unsqueeze(1), 1)

    intersection = (preds * targets_onehot).sum(dim=(0,2,3))
    union = preds.sum(dim=(0,2,3)) + targets_onehot.sum(dim=(0,2,3))

    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()

def combined_loss(preds, targets):
    return ce_loss(preds, targets) + dice_loss(preds, targets)


# Training Loop

In [27]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0

    for imgs, masks in tqdm(loader, desc="Training", leave=False):
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


#  TRAINING

In [13]:
import os

def save_checkpoint(model, optimizer, epoch, loss, path):
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "loss": loss
    }
    torch.save(checkpoint, path)


def load_checkpoint(model, optimizer, path, device):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    start_epoch = checkpoint["epoch"] + 1
    loss = checkpoint["loss"]
    print(f"Resumed from epoch {checkpoint['epoch']} | Loss: {loss:.4f}")
    return start_epoch


In [14]:
checkpoint_dir = "checkpoints_attention_unet"
os.makedirs(checkpoint_dir, exist_ok=True)


(OPTIONAL) Resume Training Flag

In [15]:
resume = False  # set True to resume
resume_path = os.path.join(checkpoint_dir, "epoch_5.pth")  # example


Initialize Model & Optimizer

In [16]:
model = AttentionUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

start_epoch = 0

if resume:
    start_epoch = load_checkpoint(model, optimizer, resume_path, device)


Final Training Loop

In [17]:
epochs = 5

for epoch in range(start_epoch, epochs):
    loss = train_epoch(model, train_loader, optimizer, device)

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss:.4f}")

    checkpoint_path = os.path.join(
        checkpoint_dir, f"epoch_{epoch+1}.pth"
    )

    save_checkpoint(
        model,
        optimizer,
        epoch,
        loss,
        checkpoint_path
    )


KeyboardInterrupt: 

# EVALUATION

In [9]:
model = AttentionUNet(in_channels=2, num_classes=2).to(device)
model.eval()


AttentionUNet(
  (enc1): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(128, 256, ker

Locate Checkpoint

In [12]:
checkpoint_path = "checkpoints_attention_unet/epoch_100.pth"  # adjust name

checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint["model_state"])
model.eval()

print("Model loaded successfully")


Model loaded successfully


C:\Users\ankit\AppData\Local\Temp\ipykernel_24480\1618753127.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device

Metrice evaluation

In [13]:
import torch

def dice_score(pred, target, eps=1e-6):
    pred = (pred > 0).float()
    target = target.float()
    intersection = (pred * target).sum()
    return (2 * intersection + eps) / (pred.sum() + target.sum() + eps)

def iou_score(pred, target, eps=1e-6):
    pred = (pred > 0).float()
    target = target.float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + eps) / (union + eps)

def precision_score(pred, target, eps=1e-6):
    pred = (pred > 0).float()
    target = target.float()
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    return (tp + eps) / (tp + fp + eps)

def recall_score(pred, target, eps=1e-6):
    pred = (pred > 0).float()
    target = target.float()
    tp = (pred * target).sum()
    fn = ((1 - pred) * target).sum()
    return (tp + eps) / (tp + fn + eps)


In [15]:
dice_list = []
iou_list = []
precision_list = []
recall_list = []

with torch.no_grad():
    for imgs, masks in train_loader:  # or val_loader
        imgs = imgs.to(device)
        masks = masks.to(device)

        outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1)

        for p, t in zip(preds, masks):
            dice_list.append(dice_score(p, t).item())
            iou_list.append(iou_score(p, t).item())
            precision_list.append(precision_score(p, t).item())
            recall_list.append(recall_score(p, t).item())

print("Dice:", sum(dice_list)/len(dice_list))
print("IoU:", sum(iou_list)/len(iou_list))
print("Precision:", sum(precision_list)/len(precision_list))
print("Recall:", sum(recall_list)/len(recall_list))


Dice: 0.47583864739631326
IoU: 0.4025336069875754
Precision: 0.6739774542278573
Recall: 0.6761257437757132


# Graph extractraction


In [33]:
import os
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [34]:
def dice_score(pred, target, eps=1e-6):
    pred = (pred > 0).float()
    target = target.float()

    intersection = (pred * target).sum()
    return (2 * intersection + eps) / (pred.sum() + target.sum() + eps)


In [35]:
eval_dataset = ISLESDataset2D(
    root_dir="D:/Capstone/Experiment 4/Datasets/ISLES-2022"
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0   # Windows-safe
)

print("Evaluation samples:", len(eval_dataset))


Indexed slices: 4827
Evaluation samples: 4827


In [36]:
model = AttentionUNet(in_channels=2, num_classes=2).to(device)
model.eval()


AttentionUNet(
  (enc1): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(2, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): DoubleConv(
    (conv): Sequential(
      (0): Conv2d(128, 256, ker

In [37]:
def evaluate_dice(model, loader, device):
    model.eval()
    dice_list = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)

            for p, t in zip(preds, masks):
                dice_list.append(dice_score(p, t).item())

    return sum(dice_list) / len(dice_list)


In [38]:
checkpoint_dir = "checkpoints_attention_unet"

checkpoint_files = sorted(
    [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")],
    key=lambda x: int(re.findall(r"\d+", x)[0])
)

print("Checkpoints found:", checkpoint_files)


Checkpoints found: ['epoch_1.pth', 'epoch_2.pth', 'epoch_3.pth', 'epoch_4.pth', 'epoch_5.pth', 'epoch_6.pth', 'epoch_7.pth', 'epoch_8.pth', 'epoch_9.pth', 'epoch_10.pth', 'epoch_11.pth', 'epoch_12.pth', 'epoch_13.pth', 'epoch_14.pth', 'epoch_15.pth', 'epoch_16.pth', 'epoch_17.pth', 'epoch_18.pth', 'epoch_19.pth', 'epoch_20.pth', 'epoch_21.pth', 'epoch_22.pth', 'epoch_23.pth', 'epoch_24.pth', 'epoch_25.pth', 'epoch_26.pth', 'epoch_27.pth', 'epoch_28.pth', 'epoch_29.pth', 'epoch_30.pth', 'epoch_31.pth', 'epoch_32.pth', 'epoch_33.pth', 'epoch_34.pth', 'epoch_35.pth', 'epoch_36.pth', 'epoch_37.pth', 'epoch_38.pth', 'epoch_39.pth', 'epoch_40.pth', 'epoch_41.pth', 'epoch_42.pth', 'epoch_43.pth', 'epoch_44.pth', 'epoch_45.pth', 'epoch_46.pth', 'epoch_47.pth', 'epoch_48.pth', 'epoch_49.pth', 'epoch_50.pth', 'epoch_51.pth', 'epoch_52.pth', 'epoch_53.pth', 'epoch_54.pth', 'epoch_55.pth', 'epoch_56.pth', 'epoch_57.pth', 'epoch_58.pth', 'epoch_59.pth', 'epoch_60.pth', 'epoch_61.pth', 'epoch_62.pth

In [39]:
dice_per_epoch = []

for ckpt in checkpoint_files:
    path = os.path.join(checkpoint_dir, ckpt)

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])

    dice = evaluate_dice(model, eval_loader, device)
    dice_per_epoch.append(dice)

    print(f"{ckpt} → Dice: {dice:.4f}")


C:\Users\ankit\AppData\Local\Temp\ipykernel_6548\729431434.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


epoch_1.pth → Dice: 0.3945


C:\Users\ankit\AppData\Local\Temp\ipykernel_6548\729431434.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


epoch_2.pth → Dice: 0.3992


C:\Users\ankit\AppData\Local\Temp\ipykernel_6548\729431434.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


...

epoch_10.pth → Dice: 0.4149


C:\Users\ankit\AppData\Local\Temp\ipykernel_6548\729431434.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


epoch_11.pth → Dice: 0.4185


C:\Users\ankit\AppData\Local\Temp\ipykernel_6548\729431434.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location=device)


KeyboardInterrupt: 